In [24]:
import json
import os
import pandas as pd
import re
import shutil
from collections import defaultdict
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from functions import *
from eval_functions import *

In [25]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)

save = True

eval_path = f"evaluate/batch1"
os.makedirs(eval_path, exist_ok=True)

### Model Output to nice JSON and Failure 

In [20]:
def process_files(model, save=save):
    input_dir = f"model_output/{model}/output/"
    output_dir = f"model_output/{model}/formatted/"
    failure_dir = f"model_output/{model}/failed/"
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(failure_dir, exist_ok=True)
    files = os.listdir(input_dir)
    len_files = len(files)
    for filename in files:
        if filename.endswith('.json'):
            input_file_path = os.path.join(input_dir, filename)
            output_file_path = os.path.join(output_dir, filename)
            failure_file_path = os.path.join(failure_dir, filename)
            try:
                file = read_json(input_file_path)
                print(f"Processing file: {filename}")
                if save:
                    shutil.copy(input_file_path, output_file_path)
                    #save_json_to_file(file, output_file_path)
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
                if save:
                    shutil.copy(input_file_path, failure_file_path)
    return len_files

In [21]:
for model in models:
    print(model)
    # 1 - Preprocess Files
    num_out_files = process_files(model, save=save)
    print(num_out_files)
    # 2 - Evaluate Files
    p2_label_path = "chia_label/p2"
    ready_path = f"model_output/{model}/ready"
    failed_model_path = f"model_output/{model}/failed_inner"
    p2_model_formatted_path = f"model_output/{model}/formatted"
    
    for path in [ready_path, failed_model_path, p2_model_formatted_path]:
        os.makedirs(path, exist_ok=True)
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.json')}
    model_files = {extract_nct_number(f): os.path.join(p2_model_formatted_path, f) for f in os.listdir(p2_model_formatted_path) if f.endswith('.json')}
    
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    
    for nct in common_ncts:
        try:
            label_data = read_json(label_files[nct])
            model_data = read_json(model_files[nct])
            print(model_data)
            label_structure = extract_logical_structure(label_data)
            model_structure = extract_logical_structure(model_data)
    
            # Try count words
            label_raw_texts = extract_raw_texts(label_data)
            model_raw_texts = extract_raw_texts(model_data)
            
            missing_texts = label_raw_texts - model_raw_texts
            
            label_words = extract_words(label_raw_texts)
            model_words = extract_words(model_raw_texts)
            missing_words = label_words - model_words
            missing_words_pct = len(missing_words) / len(label_words) * 100 if label_words else 0
            print(f"Processing NCT {nct}")
            print(round(missing_words_pct,2), "%")
            print(label_words.issubset(model_words))
            print("Label Text \n")
            print(label_words)
            print("\nModel Text \n")
            print(model_words)
            print()
            
            success_data.append({
                'NCT': nct,
                'label_AND': label_structure.get('AND', 0),
                'label_OR': label_structure.get('OR', 0),
                'label_NOT': label_structure.get('NOT', 0),
                'label_DEPTH': label_structure.get('depth', 0),
                'model_AND': model_structure.get('AND', 0),
                'model_OR': model_structure.get('OR', 0),
                'model_NOT': model_structure.get('NOT', 0),
                'model_DEPTH': model_structure.get('depth', 0),
                'diff_AND': 1 if model_structure.get('AND', 0) > label_structure.get('AND', 0) else -1 if model_structure.get('AND', 0) < label_structure.get('AND', 0) else 0,
                'diff_OR': 1 if model_structure.get('OR', 0) > label_structure.get('OR', 0) else -1 if model_structure.get('OR', 0) < label_structure.get('OR', 0) else 0,
                'diff_NOT': 1 if model_structure.get('NOT', 0) > label_structure.get('NOT', 0) else -1 if model_structure.get('NOT', 0) < label_structure.get('NOT', 0) else 0,
                'diff_DEPTH': 1 if model_structure.get('depth', 0) > label_structure.get('depth', 0) else -1 if model_structure.get('depth', 0) < label_structure.get('depth', 0) else 0,
                'num_out_files': num_out_files,
            })
    
            labels.append(label_structure)
            predictions.append(model_structure)
            save and shutil.copy(model_files[nct], os.path.join(ready_path, os.path.basename(model_files[nct])))
        except Exception as e:
            #print(f"Error processing NCT {nct}: {e}")
            save and shutil.copy(model_files[nct], os.path.join(failed_model_path, os.path.basename(model_files[nct])))
    
    df_success = pd.DataFrame(success_data).set_index('NCT')
    df_success.to_csv(eval_path+f'/{model}_eval.csv')
    true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
    predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values
    
    metrics = {}
    for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
        y_true = true_values[:, i]
        y_pred = predicted_values[:, i]
    
        diffs = y_pred - y_true
        pct_greater = (diffs > 0).sum() / len(diffs) * 100
        pct_less = (diffs < 0).sum() / len(diffs) * 100
        pct_equal = (diffs == 0).sum() / len(diffs) * 100
    
        metrics[metric] = {
            'accuracy': round(accuracy_score(y_true, y_pred), 3),
            'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
            'pct_greater': round(pct_greater, 2),
            'pct_less': round(pct_less, 2),
            'pct_equal': round(pct_equal, 2)
            # 'confusion_matrix': confusion_matrix(y_true, y_pred)
        }
        metrics_df = pd.DataFrame(metrics).T
        num_nct_files = len(df_success)
        metrics_df['num_nct_files'] = num_nct_files
        metrics_df['model_name'] = model
        metrics_df['num_files'] = anzahl_files
        # Save Metrics to CSV
        metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))    

Llama-3-8B-Instruct-Gradient-1048k_25_shot
Error processing file Llama-3-8B-Instruct-Gradient-1048k_NCT00846703_inc_25_shot.json: Expecting ',' delimiter: line 69 column 2 (char 2291)
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00862446_exc_25_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00862446_inc_25_shot.json
Error processing file Llama-3-8B-Instruct-Gradient-1048k_NCT00867958_exc_25_shot.json: Expecting ',' delimiter: line 45 column 2 (char 1542)
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00867958_inc_25_shot.json
Processing file: Llama-3-8B-Instruct-Gradient-1048k_NCT00886158_exc_25_shot.json
Error processing file Llama-3-8B-Instruct-Gradient-1048k_NCT00886158_inc_25_shot.json: Expecting ',' delimiter: line 22 column 2 (char 499)
Error processing file Llama-3-8B-Instruct-Gradient-1048k_NCT00894712_exc_25_shot.json: Expecting ',' delimiter: line 38 column 2 (char 1003)
Error processing file Llama-3-8B-Instruct-Gradient-1048k_NCT00894712_i

In [22]:
# NCT02935855_inc

In [23]:
df_success = pd.DataFrame(success_data).set_index('NCT')
df_success.to_csv(eval_path+f'/{model}_eval.csv')

df_success

,label_AND,label_OR,label_NOT,label_DEPTH,model_AND,model_OR,model_NOT,model_DEPTH,diff_AND,diff_OR,diff_NOT,diff_DEPTH,num_out_files
NCT,,,,,,,,,,,,,
NCT01680081_inc,3,1,0,7,2,1,0,5,-1,0,0,-1,300
NCT01728194_inc,7,1,0,11,4,0,0,8,-1,-1,0,-1,300
NCT00050349_exc,21,18,3,33,7,12,3,19,-1,-1,0,-1,300
NCT01084993_exc,3,3,1,11,4,1,1,9,1,-1,0,-1,300
NCT01000155_exc,15,8,3,33,10,9,0,19,-1,1,-1,-1,300
...,...,...,...,...,...,...,...,...,...,...,...,...,...
NCT01567605_exc,11,5,2,23,3,3,0,9,-1,-1,-1,-1,300
NCT01175044_inc,1,0,0,3,1,0,0,3,0,0,0,0,300
NCT00440245_inc,1,1,0,5,0,1,0,3,-1,0,0,-1,300


In [18]:
true_values = df_success[['label_AND', 'label_OR', 'label_NOT', 'label_DEPTH']].values
predicted_values = df_success[['model_AND', 'model_OR', 'model_NOT', 'model_DEPTH']].values

metrics = {}
for i, metric in enumerate(['AND', 'OR', 'NOT', 'DEPTH']):
    y_true = true_values[:, i]
    y_pred = predicted_values[:, i]

    diffs = y_pred - y_true
    pct_greater = (diffs > 0).sum() / len(diffs) * 100
    pct_less = (diffs < 0).sum() / len(diffs) * 100
    pct_equal = (diffs == 0).sum() / len(diffs) * 100


    metrics[metric] = {
        'accuracy': round(accuracy_score(y_true, y_pred), 3),
        'precision': round(precision_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'recall': round(recall_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'f1_score': round(f1_score(y_true, y_pred, average='weighted', zero_division=0), 3),
        'pct_greater': round(pct_greater, 2),
        'pct_less': round(pct_less, 2),
        'pct_equal': round(pct_equal, 2)
       # 'confusion_matrix': confusion_matrix(y_true, y_pred)
    }
    metrics_df = pd.DataFrame(metrics).T  
    num_nct_files = len(df_success)
    metrics_df['num_out_files'] = num_nct_files
    metrics_df['num_nct_files'] = num_nct_files
    metrics_df['model_name'] = model
    # Save Metrics to CSV
    metrics_df.to_csv(os.path.join(eval_path, f'{model}_metrics_summary.csv'))

print(f"{len(df_success)} Daten mit {model}")
for metric, values in metrics.items():
    print(f"Metrics for {metric}:")
    print(f"  Accuracy: {values['accuracy']}")
    print(f"  Precision: {values['precision']}")
    print(f"  Recall: {values['recall']}")
    print(f"  F1 Score: {values['f1_score']}")

    print(f"  % Greater: {values['pct_greater']}")
    print(f"  % Less: {values['pct_less']}")
    print(f"  % Equal: {values['pct_equal']}")
    print()
    #print(f"  Confusion Matrix:\n{values['confusion_matrix']}\n")

264 Daten mit Llama-3-8B-Instruct_4_shot_temp_1
Metrics for AND:
  Accuracy: 0.121
  Precision: 0.102
  Recall: 0.121
  F1 Score: 0.103
  % Greater: 17.8
  % Less: 70.08
  % Equal: 12.12

Metrics for OR:
  Accuracy: 0.246
  Precision: 0.277
  Recall: 0.246
  F1 Score: 0.256
  % Greater: 42.05
  % Less: 33.33
  % Equal: 24.62

Metrics for NOT:
  Accuracy: 0.625
  Precision: 0.571
  Recall: 0.625
  F1 Score: 0.579
  % Greater: 9.85
  % Less: 27.65
  % Equal: 62.5

Metrics for DEPTH:
  Accuracy: 0.121
  Precision: 0.109
  Recall: 0.121
  F1 Score: 0.105
  % Greater: 15.53
  % Less: 72.35
  % Equal: 12.12



In [19]:
matching_rows = df_success[df_success['label_AND'] == df_success['model_AND']][['label_AND', 'model_AND']]
matching_rows

,label_AND,model_AND
NCT,,
NCT00718952_inc,7,7
NCT00480129_inc,2,2
NCT01501201_inc,4,4
NCT01391780_inc,1,1
NCT00846703_exc,6,6
NCT00787254_inc,2,2
NCT00404495_exc,3,3
NCT00425789_inc,3,3
NCT00959569_inc,3,3
